In [ ]:

import pandas as pd
import numpy as np
import zipfile
import os

In [ ]:
zip_path = "collections_30k_dataset.zip"

data = {}

with zipfile.ZipFile(zip_path, "r") as z:
    for file in z.namelist():
        if file.endswith(".csv"):
            name = file.replace(".csv", "")
            data[name] = pd.read_csv(z.open(file), low_memory=False)

print("Datasets loaded:", len(data))

for name, df in data.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
# Dataset quality report
quality_report = []

for name, df in data.items():

    total_cells = df.shape[0] * df.shape[1]

    quality_report.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate Rows": df.duplicated().sum(),
        "Missing Cells": df.isna().sum().sum(),
        "Missing %": round(
            df.isna().sum().sum() / total_cells * 100, 2
        ) if total_cells else 0
    })

quality_df = pd.DataFrame(quality_report)

display(quality_df.sort_values("Missing %", ascending=False))


In [ ]:

# Candidate primary keys based on the data dictionary

primary_keys = {
    "borrowers": "borrower_id",
    "accounts": "account_id",
    "agents": "agent_id",
    "agent_sessions": "session_id",
    "campaigns": "campaign_id",
    "daily_targeting": "target_id",
    "calls": "call_id",
    "call_attempts": "attempt_id",
    "call_dispositions": "disposition_id",
    "whatsapp_events": "whatsapp_event_id",
    "sms_events": "sms_event_id",
    "field_visits": "visit_id",
    "promises_to_pay": "ptp_id",
    "payments": "payment_id",
    "complaints": "complaint_id",
    "account_status_history": "history_id"
}

for table, key in primary_keys.items():
    df = data[table]

    print(
        table,
        "| Rows:", len(df),
        "| Unique IDs:", df[key].nunique(),
        "| Missing IDs:", df[key].isna().sum(),
        "| Repeated IDs:", df[key].duplicated().sum()
    )


In [ ]:

# Identify date columns and their coverage

for name, df in data.items():

    for col in df.columns:
        if any(x in col.lower() for x in
               ["date", "time", "timestamp", "_at"]):

            dates = pd.to_datetime(
                df[col], errors="coerce", utc=True
            )

            if dates.notna().any():
                print(
                    name, "|", col,
                    "| Start:", dates.min(),
                    "| End:", dates.max(),
                    "| Missing:", dates.isna().sum()
                )


In [ ]:

# Foreign-key checks

relationships = [
    ("calls", "account_id", "accounts", "account_id"),
    ("payments", "account_id", "accounts", "account_id"),
    ("promises_to_pay", "account_id", "accounts", "account_id"),
    ("call_attempts", "account_id", "accounts", "account_id"),
    ("daily_targeting", "account_id", "accounts", "account_id"),
    ("whatsapp_events", "account_id", "accounts", "account_id"),
    ("sms_events", "account_id", "accounts", "account_id"),
    ("field_visits", "account_id", "accounts", "account_id"),
    ("complaints", "account_id", "accounts", "account_id")
]

for child, child_col, parent, parent_col in relationships:

    valid_ids = set(data[parent][parent_col].dropna())

    orphan_count = (
        ~data[child][child_col].isin(valid_ids)
    ).sum()

    print(
        f"{child} -> {parent}: "
        f"{orphan_count} unmatched records"
    )
